# Иерархическая классификация русских запросов: domain → field → subfield

**Постановка задачи.** Короткий русский запрос → научная область из таксономии OpenAlex на трёх уровнях иерархии:
- **domain** (4 класса): Health Sciences, Life Sciences, Physical Sciences, Social Sciences
- **field** (26 классов): Medicine, Computer Science, Mathematics, …
- **subfield** (193 класса): Cardiology and Cardiovascular Medicine, Artificial Intelligence, Algebra and Number Theory, …

**Архитектура.** `multilingual-e5-large-instruct` как энкодер + linear-классификатор на subfield (193 класса). Field и domain выводятся из subfield через детерминированный маппинг (иерархия строгая: один subfield → один field → один domain).

**Почему один классификатор, а не мульти-таск.** Иерархия консистентна, поэтому subfield однозначно определяет field и domain. Один CE-loss на самом мелком уровне — самый чистый сигнал; домен/филд легко получить лукапом на инференсе. Добавлять отдельные головы под field/domain — это лишние гиперпараметры (веса лоссов) ради зеркальной информации.

**Метрики — классификации, не ранжирования.** Accuracy, precision, recall, macro-F1, weighted-F1 на каждом из трёх уровней + per-class report + confusion matrix на уровне domain.

**Среда.** A100 (40GB+).


## 1. Импорты

In [ ]:
import json
import math
import random
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from transformers import (
    AutoModel, AutoTokenizer,
    Trainer, TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Конфигурация

In [ ]:
CFG = {
    'data_path': '../data/dataset.csv',
    'model_name': 'intfloat/multilingual-e5-large-instruct',
    'output_dir': './e5_subfield_classifier',
    'checkpoint_dir': './e5_subfield_checkpoint_final',

    # Сплит (по уникальным запросам, чтобы избежать утечки)
    'val_size': 0.10,
    'test_size': 0.10,

    # Тренировка
    'batch_size': 32,
    'eval_batch_size': 64,
    'gradient_accumulation_steps': 2,    # эффективный batch = 64
    'num_epochs': 4,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'max_seq_length': 64,                # короткие запросы
    'fp16': True,
    'dropout': 0.1,
    'label_smoothing': 0.05,             # помогает на 193 классах с дисбалансом

    # E5 prefix для query
    'query_prefix': 'query: ',
}

Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
Path(CFG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2, ensure_ascii=False))

## 3. Загрузка данных и иерархии

Дедуплицируем по запросу: исходный `dataset.csv` содержит 100K строк (query, topic), но многие запросы повторяются с разными старыми topic-метками. После маппинга `query → concept → (domain, field, subfield)` все дубликаты получают одинаковые новые метки, поэтому имеем 18 852 уникальных пары `(query, subfield)`.

In [ ]:
df = pd.read_csv(CFG['data_path'])
print(f'rows: {len(df):,}  |  cols: {list(df.columns)}')

# Дедуп по query — каждая (query, domain, field, subfield) уникальна
df = df[['query', 'domain', 'field', 'subfield']].drop_duplicates(subset='query').reset_index(drop=True)
print(f'после дедупа по query: {len(df):,}')

# Иерархия (subfield -> field -> domain)
hier = df[['subfield','field','domain']].drop_duplicates().sort_values(['domain','field','subfield']).reset_index(drop=True)
print(f'\nИерархия: {df["domain"].nunique()} domain, {df["field"].nunique()} field, {df["subfield"].nunique()} subfield')

# Маппинги label <-> id (фиксированный порядок по алфавиту для воспроизводимости)
domains = sorted(df['domain'].unique())
fields = sorted(df['field'].unique())
subfields = sorted(df['subfield'].unique())

domain2id = {d: i for i, d in enumerate(domains)}
field2id = {f: i for i, f in enumerate(fields)}
subfield2id = {s: i for i, s in enumerate(subfields)}
id2subfield = {i: s for s, i in subfield2id.items()}

# Lookup-таблицы для вывода field/domain из subfield
sf_to_f = dict(zip(hier['subfield'], hier['field']))
sf_to_d = dict(zip(hier['subfield'], hier['domain']))
sf_id_to_f_id = np.array([field2id[sf_to_f[id2subfield[i]]] for i in range(len(subfields))])
sf_id_to_d_id = np.array([domain2id[sf_to_d[id2subfield[i]]] for i in range(len(subfields))])

print(f'\nПримеры маппинга (subfield_id -> field_id -> domain_id):')
for i in [0, 50, 100, 150, 192]:
    print(f'  {i:3d} {id2subfield[i]!r:55s} -> {fields[sf_id_to_f_id[i]]} -> {domains[sf_id_to_d_id[i]]}')

## 4. Распределение классов

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

dc = df['domain'].value_counts()
axes[0].barh(dc.index, dc.values, color='steelblue', edgecolor='black')
axes[0].set_title(f'Domain ({len(dc)} классов)'); axes[0].set_xlabel('запросов'); axes[0].invert_yaxis()

fc = df['field'].value_counts()
axes[1].barh(fc.index, fc.values, color='seagreen', edgecolor='black')
axes[1].set_title(f'Field ({len(fc)} классов)'); axes[1].set_xlabel('запросов'); axes[1].invert_yaxis()
axes[1].tick_params(axis='y', labelsize=7)

sc = df['subfield'].value_counts()
axes[2].hist(sc.values, bins=50, color='indianred', edgecolor='black')
axes[2].set_title(f'Subfield: распределение размеров классов ({len(sc)} классов)')
axes[2].set_xlabel('запросов на класс'); axes[2].set_ylabel('классов')
axes[2].axvline(sc.median(), color='black', linestyle='--', label=f'median={int(sc.median())}')
axes[2].legend()

plt.tight_layout(); plt.show()

print(f'\nSubfield:  min={sc.min()}  median={int(sc.median())}  mean={sc.mean():.1f}  max={sc.max()}')
print(f'Subfield-классов с <5 примеров: {(sc < 5).sum()}')

## 5. Стратифицированный сплит по subfield

Стратифицируем по самому мелкому уровню (subfield) — это автоматически обеспечивает баланс на field и domain.

In [ ]:
# Для классов с <2 примерами стратификация невозможна — fallback к non-stratified
sf_counts = df['subfield'].value_counts()
rare = sf_counts[sf_counts < 2].index
print(f'Subfield-классов с <2 примеров (вне страт. сплита): {len(rare)}')

if len(rare) > 0:
    df_strat = df[~df['subfield'].isin(rare)].reset_index(drop=True)
    df_rare = df[df['subfield'].isin(rare)].reset_index(drop=True)
else:
    df_strat = df; df_rare = df.iloc[:0]

trainval, test = train_test_split(df_strat, test_size=CFG['test_size'],
                                   random_state=SEED, stratify=df_strat['subfield'])
val_rel = CFG['val_size'] / (1 - CFG['test_size'])
train, val = train_test_split(trainval, test_size=val_rel,
                               random_state=SEED, stratify=trainval['subfield'])

# Редкие классы кладём в train
train = pd.concat([train, df_rare], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
val = val.reset_index(drop=True); test = test.reset_index(drop=True)

print(f'Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}')
print(f'Subfield-классов  train={train["subfield"].nunique()}  val={val["subfield"].nunique()}  test={test["subfield"].nunique()}')

## 6. Токенизация и Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
print(f'Tokenizer: {CFG["model_name"]}')

class QueryDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, prefix):
        self.queries = (prefix + df['query']).tolist()
        self.labels = df['subfield'].map(subfield2id).values.astype(np.int64)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, i):
        enc = self.tok(self.queries[i], truncation=True, max_length=self.max_len,
                       padding=False, return_tensors=None)
        enc['labels'] = int(self.labels[i])
        return enc

train_ds = QueryDataset(train, tokenizer, CFG['max_seq_length'], CFG['query_prefix'])
val_ds = QueryDataset(val, tokenizer, CFG['max_seq_length'], CFG['query_prefix'])
test_ds = QueryDataset(test, tokenizer, CFG['max_seq_length'], CFG['query_prefix'])

print(f'\nПример токенизированной строки:')
sample = train_ds[0]
print(f'  text:      {tokenizer.decode(sample["input_ids"])!r}')
print(f'  label:     {sample["labels"]} ({id2subfield[sample["labels"]]})')
print(f'  n_tokens:  {len(sample["input_ids"])}')

## 7. Модель: E5 encoder + linear-классификатор

Mean-pooling по последнему скрытому слою (стандартный для E5), dropout, линейная голова на 193 класса.

In [ ]:
N_SUBFIELDS = len(subfields)

class E5Classifier(nn.Module):
    def __init__(self, model_name, n_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, n_classes)

    @staticmethod
    def mean_pool(last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        logits = self.classifier(self.dropout(pooled))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, label_smoothing=CFG['label_smoothing'])
        return {'loss': loss, 'logits': logits}

model = E5Classifier(CFG['model_name'], N_SUBFIELDS, dropout=CFG['dropout'])
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model:   {CFG["model_name"]}')
print(f'Hidden:  {model.encoder.config.hidden_size}')
print(f'Heads:   subfield ({N_SUBFIELDS} классов) — field/domain выводятся через lookup')
print(f'Params:  {n_params/1e6:.1f}M (trainable: {n_trainable/1e6:.1f}M)')

## 8. Метрики

`compute_metrics` для Trainer считает основное на subfield. После тренировки на test делаем полный отчёт на всех трёх уровнях.

In [ ]:
def metrics_at_level(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    return {
        'accuracy': float(acc),
        'precision_macro': float(p_macro),
        'recall_macro': float(r_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(p_w),
        'recall_weighted': float(r_w),
        'f1_weighted': float(f1_w),
    }

def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    sf_pred = logits.argmax(axis=-1)
    f_pred = sf_id_to_f_id[sf_pred]
    d_pred = sf_id_to_d_id[sf_pred]
    sf_true = labels
    f_true = sf_id_to_f_id[sf_true]
    d_true = sf_id_to_d_id[sf_true]

    out = {}
    for level, t, p in [('subfield', sf_true, sf_pred),
                         ('field', f_true, f_pred),
                         ('domain', d_true, d_pred)]:
        m = metrics_at_level(t, p)
        for k, v in m.items():
            out[f'{level}_{k}'] = v
    return out

## 9. Тренировка

In [ ]:
training_args = TrainingArguments(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['num_epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    per_device_eval_batch_size=CFG['eval_batch_size'],
    gradient_accumulation_steps=CFG['gradient_accumulation_steps'],
    learning_rate=CFG['learning_rate'],
    weight_decay=CFG['weight_decay'],
    warmup_ratio=CFG['warmup_ratio'],
    fp16=CFG['fp16'],
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='subfield_f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
)

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics_for_trainer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Training steps per epoch: {len(train_ds) // (CFG["batch_size"] * CFG["gradient_accumulation_steps"]):,}')
print(f'Эффективный batch: {CFG["batch_size"] * CFG["gradient_accumulation_steps"]}')

In [ ]:
train_result = trainer.train()
print('\n=== Training complete ===')
print(json.dumps(train_result.metrics, indent=2))

## 10. Кривые обучения

In [ ]:
log = trainer.state.log_history
train_steps, train_losses = [], []
eval_epochs, eval_metrics = [], []
for entry in log:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step']); train_losses.append(entry['loss'])
    if 'eval_loss' in entry:
        eval_epochs.append(entry.get('epoch', 0)); eval_metrics.append(entry)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(train_steps, train_losses, alpha=0.4, color='steelblue', label='train loss')
if len(train_losses) >= 20:
    w = max(5, len(train_losses) // 50)
    axes[0].plot(train_steps, pd.Series(train_losses).rolling(w, min_periods=1).mean(),
                 color='navy', linewidth=2, label=f'rolling mean (w={w})')
axes[0].set_xlabel('step'); axes[0].set_ylabel('loss'); axes[0].set_title('Train loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

if eval_metrics:
    for level, color, marker in [('domain','#1f77b4','o'), ('field','#2ca02c','s'), ('subfield','#d62728','^')]:
        axes[1].plot(eval_epochs, [m[f'eval_{level}_f1_macro'] for m in eval_metrics],
                     marker=marker, color=color, label=f'{level} F1-macro')
        axes[1].plot(eval_epochs, [m[f'eval_{level}_accuracy'] for m in eval_metrics],
                     marker=marker, color=color, linestyle='--', alpha=0.5, label=f'{level} accuracy')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('score'); axes[1].set_title('Val metrics')
axes[1].legend(loc='lower right', fontsize=8); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(Path(CFG['output_dir']) / 'training_curves.png', dpi=110, bbox_inches='tight')
plt.show()

## 11. Оценка на test (полный отчёт по трём уровням)

In [ ]:
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
if isinstance(logits, tuple): logits = logits[0]

sf_pred = logits.argmax(axis=-1)
sf_true = pred_out.label_ids
f_pred = sf_id_to_f_id[sf_pred]; f_true = sf_id_to_f_id[sf_true]
d_pred = sf_id_to_d_id[sf_pred]; d_true = sf_id_to_d_id[sf_true]

print('=' * 70)
print('TEST METRICS — все три уровня иерархии')
print('=' * 70)
test_summary = {}
for level, t, p in [('domain', d_true, d_pred),
                     ('field', f_true, f_pred),
                     ('subfield', sf_true, sf_pred)]:
    m = metrics_at_level(t, p)
    test_summary[level] = m
    print(f'\n{level.upper()}:')
    for k, v in m.items():
        print(f'  {k:25s} {v:.4f}')

with open(Path(CFG['output_dir']) / 'test_metrics.json', 'w') as f:
    json.dump(test_summary, f, indent=2)

In [ ]:
# Per-class отчёт на domain (4 класса — компактно)
print('=== DOMAIN: per-class report ===')
print(classification_report(d_true, d_pred, target_names=domains, digits=4, zero_division=0))

In [ ]:
# Confusion matrix на domain
cm = confusion_matrix(d_true, d_pred, labels=list(range(len(domains))))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(domains))); ax.set_yticks(range(len(domains)))
ax.set_xticklabels(domains, rotation=30, ha='right')
ax.set_yticklabels(domains)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title('Confusion matrix — domain (нормированная по строкам)')
for i in range(len(domains)):
    for j in range(len(domains)):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i, j]:.2f}\n({cm[i, j]})', ha='center', va='center', color=color, fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(Path(CFG['output_dir']) / 'confusion_matrix_domain.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class отчёт на field (26 классов)
print('=== FIELD: per-class report ===')
print(classification_report(f_true, f_pred, target_names=fields, digits=4, zero_division=0))

In [ ]:
# Топ-20 худших subfield-классов по F1 (там где модель чаще всего ошибается)
sf_report = classification_report(sf_true, sf_pred, target_names=subfields, digits=4,
                                   zero_division=0, output_dict=True)
sf_rows = [(name, sf_report[name]) for name in subfields if name in sf_report]
sf_rows.sort(key=lambda x: x[1]['f1-score'])

print('=== SUBFIELD: 20 классов с худшим F1 ===')
print(f'{"subfield":55s} {"F1":>7s} {"prec":>7s} {"recall":>7s} {"support":>8s}')
for name, m in sf_rows[:20]:
    print(f'{name[:55]:55s} {m["f1-score"]:7.4f} {m["precision"]:7.4f} {m["recall"]:7.4f} {int(m["support"]):>8d}')

print('\n=== SUBFIELD: 10 классов с лучшим F1 ===')
for name, m in sf_rows[-10:][::-1]:
    print(f'{name[:55]:55s} {m["f1-score"]:7.4f} {m["precision"]:7.4f} {m["recall"]:7.4f} {int(m["support"]):>8d}')

## 12. Сохранение чекпоинта

In [ ]:
from datetime import datetime
import shutil

ckpt = Path(CFG['checkpoint_dir'])
ckpt.mkdir(parents=True, exist_ok=True)

# 1. Веса модели (encoder + classifier head) и токенизатор
torch.save(trainer.model.state_dict(), ckpt / 'model.pt')
trainer.model.encoder.save_pretrained(ckpt / 'encoder')
tokenizer.save_pretrained(ckpt / 'tokenizer')
print(f'  [1/5] model.pt + encoder/ + tokenizer/  ->  {ckpt}')

# 2. Маппинги меток + иерархия
labels_payload = {
    'domains': domains,
    'fields': fields,
    'subfields': subfields,
    'domain2id': domain2id,
    'field2id': field2id,
    'subfield2id': subfield2id,
    'sf_id_to_f_id': sf_id_to_f_id.tolist(),
    'sf_id_to_d_id': sf_id_to_d_id.tolist(),
    'hierarchy': [{'subfield': r['subfield'], 'field': r['field'], 'domain': r['domain']}
                  for _, r in hier.iterrows()],
}
with open(ckpt / 'labels.json', 'w', encoding='utf-8') as f:
    json.dump(labels_payload, f, ensure_ascii=False, indent=2)
print(f'  [2/5] labels.json  (4 domain / {len(fields)} field / {len(subfields)} subfield)')

# 3. Конфиг + промпт
with open(ckpt / 'config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'cfg': CFG,
        'model_name_base': CFG['model_name'],
        'hidden_size': trainer.model.encoder.config.hidden_size,
        'n_subfields': len(subfields),
        'query_prefix': CFG['query_prefix'],
        'max_seq_length': CFG['max_seq_length'],
        'pooling': 'mean',
        'saved_at': datetime.now().isoformat(timespec='seconds'),
    }, f, ensure_ascii=False, indent=2)
print(f'  [3/5] config.json')

# 4. Метрики и история
with open(ckpt / 'test_metrics.json', 'w') as f:
    json.dump(test_summary, f, indent=2)
with open(ckpt / 'training_history.json', 'w') as f:
    json.dump({
        'train_loss': [{'step': s, 'loss': l} for s, l in zip(train_steps, train_losses)],
        'val_metrics_per_epoch': eval_metrics,
        'test_metrics': test_summary,
        'split_sizes': {'train': len(train), 'val': len(val), 'test': len(test)},
    }, f, indent=2)
print(f'  [4/5] test_metrics.json + training_history.json')

# 5. Кривые + confusion matrix
for fname in ['training_curves.png', 'confusion_matrix_domain.png']:
    src = Path(CFG['output_dir']) / fname
    if src.exists():
        shutil.copy(src, ckpt / fname)
print(f'  [5/5] training_curves.png + confusion_matrix_domain.png')

# README
readme = f'''# E5 Subfield Classifier

`{CFG["model_name"]}` дообученный на классификацию русских запросов в OpenAlex-иерархию:
**4 domain → {len(fields)} field → {len(subfields)} subfield**.

## Метрики на test ({len(test):,} запросов)

| Уровень  | Accuracy | F1-macro | F1-weighted |
|----------|----------|----------|-------------|
| domain   | {test_summary["domain"]["accuracy"]:.4f}   | {test_summary["domain"]["f1_macro"]:.4f}   | {test_summary["domain"]["f1_weighted"]:.4f}      |
| field    | {test_summary["field"]["accuracy"]:.4f}   | {test_summary["field"]["f1_macro"]:.4f}   | {test_summary["field"]["f1_weighted"]:.4f}      |
| subfield | {test_summary["subfield"]["accuracy"]:.4f}   | {test_summary["subfield"]["f1_macro"]:.4f}   | {test_summary["subfield"]["f1_weighted"]:.4f}      |

Field и domain выводятся из predicted subfield через `labels.json::sf_id_to_f_id` и `sf_id_to_d_id`.

## Содержимое
- `model.pt` — full state_dict (encoder + classifier head)
- `encoder/` — HF-формат базового энкодера
- `tokenizer/` — токенизатор
- `labels.json` — маппинги меток + иерархия subfield→field→domain
- `config.json` — гиперпараметры + промпт-формат
- `test_metrics.json`, `training_history.json` — метрики
- `training_curves.png`, `confusion_matrix_domain.png` — графики

## Минимальный инференс — см. ячейку 14 ноутбука.
Сохранено: {datetime.now().isoformat(timespec="seconds")}
'''
with open(ckpt / 'README.md', 'w', encoding='utf-8') as f:
    f.write(readme)

total = sum(p.stat().st_size for p in ckpt.rglob('*') if p.is_file())
print(f'\nЧекпоинт: {ckpt.absolute()}  |  размер: {total/1e9:.2f} GB')

## 13. Интерактивная классификация

In [ ]:
def classify(query, top_k=5):
    '''Возвращает top-k subfield + выведенные field/domain.'''
    text = CFG['query_prefix'] + query
    enc = tokenizer(text, truncation=True, max_length=CFG['max_seq_length'],
                    padding=True, return_tensors='pt').to(DEVICE)
    trainer.model.eval()
    with torch.no_grad():
        out = trainer.model(**enc)
        probs = F.softmax(out['logits'], dim=-1).squeeze(0).cpu().numpy()
    top_idx = probs.argsort()[-top_k:][::-1]
    results = []
    for i in top_idx:
        sf = subfields[i]
        f = fields[sf_id_to_f_id[i]]
        d = domains[sf_id_to_d_id[i]]
        results.append({'subfield': sf, 'field': f, 'domain': d, 'prob': float(probs[i])})
    return results

# Примеры
for q in [
    'квантовая запутанность простыми словами',
    'как лечить мигрень',
    'нейросети для генерации изображений',
    'причины распада ссср',
    'правила употребления артиклей в английском',
    'теорема ферма доказательство',
    'сорта пшеницы для засушливых регионов',
    'cap-теорема для распределённых баз',
]:
    preds = classify(q, top_k=3)
    print(f'\n{q!r}')
    for r in preds:
        print(f'  [{r["prob"]:.3f}]  {r["domain"]}  >  {r["field"]}  >  {r["subfield"]}')

## 14. Standalone-инференс из чекпоинта

In [ ]:
# Запускается в чистом ноутбуке — нужны torch + transformers
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModel, AutoTokenizer

CHECKPOINT = Path('./e5_subfield_checkpoint_final')
DEVICE_INF = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg_loaded = json.load(open(CHECKPOINT / 'config.json', encoding='utf-8'))
labels_loaded = json.load(open(CHECKPOINT / 'labels.json', encoding='utf-8'))

class _E5Classifier(nn.Module):
    def __init__(self, encoder_path, n_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_path)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, n_classes)
    @staticmethod
    def mean_pool(h, m):
        m = m.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(self.mean_pool(out.last_hidden_state, attention_mask)))

inf_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT / 'tokenizer')
inf_model = _E5Classifier(CHECKPOINT / 'encoder', n_classes=cfg_loaded['n_subfields'])
inf_model.load_state_dict(torch.load(CHECKPOINT / 'model.pt', map_location=DEVICE_INF))
inf_model.to(DEVICE_INF).eval()

inf_subfields = labels_loaded['subfields']
inf_fields = labels_loaded['fields']
inf_domains = labels_loaded['domains']
inf_sf_to_f = labels_loaded['sf_id_to_f_id']
inf_sf_to_d = labels_loaded['sf_id_to_d_id']

def classify_loaded(query, top_k=5):
    text = cfg_loaded['query_prefix'] + query
    enc = inf_tokenizer(text, truncation=True, max_length=cfg_loaded['max_seq_length'],
                         padding=True, return_tensors='pt').to(DEVICE_INF)
    with torch.no_grad():
        logits = inf_model(**{k: v for k, v in enc.items() if k in ('input_ids','attention_mask')})
        probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    top_idx = probs.argsort()[-top_k:][::-1]
    return [{
        'subfield': inf_subfields[int(i)],
        'field': inf_fields[inf_sf_to_f[int(i)]],
        'domain': inf_domains[inf_sf_to_d[int(i)]],
        'prob': float(probs[int(i)]),
    } for i in top_idx]

print(f'Загружено: {len(inf_subfields)} subfield / {len(inf_fields)} field / {len(inf_domains)} domain')
for q in ['квантовая запутанность простыми словами', 'как лечить мигрень']:
    print(f'\n{q!r}')
    for r in classify_loaded(q, top_k=3):
        print(f'  [{r["prob"]:.3f}]  {r["domain"]}  >  {r["field"]}  >  {r["subfield"]}')